In [101]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd
from collections import Counter

import numpy as np
from torch.utils.data import Dataset, DataLoader

import torch
import torch.nn as nn

import pandas as pd
import numpy as np
import torch.optim as optim
import torch

import utils

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


cuda


In [102]:
period_df = pd.read_csv('../../Data_processed/for_period_gen.csv')

period_df['period'] = period_df['period'].apply(lambda x: round(x))
period_df = period_df[period_df['period'] > 0]

print(period_df.head())
print(period_df['period'].value_counts())

       LCLid      mean  median       std    min    max  gradient  period
1  MAC000003  0.397263   0.166  0.614692  0.007  3.921  0.176724      28
2  MAC000005  0.095425   0.041  0.122226  0.010  1.979  0.057801       8
3  MAC000007  0.197888   0.115  0.234394  0.015  3.784  0.105829      43
4  MAC000008  0.363184   0.296  0.241725  0.000  3.581  0.100872       3
5  MAC000009  0.179006   0.136  0.169858  0.027  2.435  0.090854       7
period
3     1055
4      481
48     423
5      265
47     261
24     240
6      187
23     158
7      140
8      112
2       89
9       82
22      79
10      74
46      73
21      69
20      63
11      54
45      46
19      44
14      42
44      36
25      35
12      35
16      32
18      32
13      30
26      27
17      25
43      23
30      21
42      20
40      19
28      18
36      18
38      18
41      17
39      16
27      16
15      16
32      13
29      13
35      12
33      11
34       9
31       7
37       6
Name: count, dtype: int64


In [103]:
class PeriodDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Get the input features
        features = self.dataframe.iloc[idx, 1:-1].values.astype('float32')
        # Get the target (period)
        target = self.dataframe.iloc[idx, -1].astype('int64') - 2
        return torch.tensor(features), torch.tensor(target)
    
dataset = PeriodDataset(period_df)

dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Classification

In [104]:
class PeriodClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(PeriodClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [107]:
# Hyperparameters
input_dim = 6  # Number of features
hidden_dim = 128  # Number of hidden units
output_dim = 47  # Number of classes (48-2+1)
learning_rate = 1e-4
num_epochs = 100

# Initialize the dataset and dataloader
dataset = PeriodDataset(period_df)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Initialize the model, loss function, and optimizer
model = PeriodClassifier(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()  # Suitable for classification tasks
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [108]:
for epoch in range(num_epochs):
    model.train()
    for features, target in dataloader:
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/100], Loss: 3.8281
Epoch [2/100], Loss: 3.7137
Epoch [3/100], Loss: 3.5994
Epoch [4/100], Loss: 3.6225
Epoch [5/100], Loss: 3.4169
Epoch [6/100], Loss: 3.3153
Epoch [7/100], Loss: 3.2523
Epoch [8/100], Loss: 3.1332
Epoch [9/100], Loss: 3.3421
Epoch [10/100], Loss: 3.2985


KeyboardInterrupt: 

# Using VAE

In [95]:
class period_Encoder(nn.Module):

    def __init__(self, hidden_size, output_size):
        super(period_Encoder, self).__init__()

        self.hidden_size = hidden_size

        self.fc = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
            nn.ReLU()
        )

    def forward(self, x):

        out = self.fc(x)

        return out

class period_Decoder(nn.Module):
    def __init__(self, latent_size, hidden_size, output_size=1):
        super(period_Decoder, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(6 + latent_size, hidden_size),  # 6 input features + latent size
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
            nn.ReLU()
        )

    def forward(self, z, original_features):
        # Concatenate the latent vector and original features
        x = torch.cat([z, original_features], dim=1)
        out = self.fc(x)
        return out


    
class period_VAE(nn.Module):
    def __init__(self, input_size=6, hidden_size = 1024, latent_size=256):
        super(period_VAE, self).__init__()

        # Define the encoder and decoder
        self.encoder = period_Encoder(hidden_size=hidden_size, output_size=latent_size * 2)  # output_size is 2*latent_size for mu and logvar
        self.decoder = period_Decoder(latent_size=latent_size, hidden_size=hidden_size, output_size=1)  # Predict the period as a single value

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, target, original_features):
        # Encode the target to get mu and logvar
        encoder_output = self.encoder(target.unsqueeze(1))  # Add a dimension for target since it's a single value
        mu, logvar = torch.chunk(encoder_output, 2, dim=1)  # Split the encoder output into mu and logvar

        # Reparameterize to get the latent vector
        z = self.reparameterize(mu, logvar)

        # Decode the latent vector and original features to reconstruct the period
        reconstruction = self.decoder(z, original_features)

        return reconstruction, mu, logvar



In [96]:
model = period_VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
epochs = 1000

In [97]:
for epoch in range(epochs):
    epoch_loss = 0.0
    for i, (features, target) in enumerate(dataloader):
        features, target = features.to(device), target.to(device)
        
        optimizer.zero_grad()
        reconstruction, mu, logvar = model(target.float(), features.float())
        
        loss = utils.vae_loss_function(reconstruction, target.float(), mu, logvar)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

    # Calculate the average loss for the epoch
    avg_loss = epoch_loss / len(dataloader)
    if epoch % 10 == 0:
        print(f'Epoch: {epoch+1}, Average Loss: {avg_loss:.4f}')

c:\Users\edwar\Desktop\Advanced Project\VAE\Code\VAE\utils.py:22: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')
c:\Users\edwar\Desktop\Advanced Project\VAE\Code\VAE\utils.py:22: UserWarning: Using a target size (torch.Size([82])) that is different to the input size (torch.Size([82, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')


Epoch: 1, Average Loss: 33902857347982.2227
Epoch: 11, Average Loss: 397062822115.5555
Epoch: 21, Average Loss: 144343929969.7778
Epoch: 31, Average Loss: 73666437575.1111
Epoch: 41, Average Loss: 43819890915.5556
Epoch: 51, Average Loss: 27714826581.3333
Epoch: 61, Average Loss: 18972498119.1111
Epoch: 71, Average Loss: 13637304433.7778
Epoch: 81, Average Loss: 9927862449.7778
Epoch: 91, Average Loss: 7416912597.3333
Epoch: 101, Average Loss: 5684017973.3333
Epoch: 111, Average Loss: 4426665841.7778
Epoch: 121, Average Loss: 3490813393.7778
Epoch: 131, Average Loss: 2779738805.3333
Epoch: 141, Average Loss: 2230530295.1111
Epoch: 151, Average Loss: 1800421678.2222
Epoch: 161, Average Loss: 1460715703.1111
Epoch: 171, Average Loss: 1189423329.7778
Epoch: 181, Average Loss: 972024080.0000
Epoch: 191, Average Loss: 796382784.8889
Epoch: 201, Average Loss: 654039324.4444
Epoch: 211, Average Loss: 538145014.2222
Epoch: 221, Average Loss: 443655921.3333
Epoch: 231, Average Loss: 366327717.3

KeyboardInterrupt: 